# Lab assignment 2 Tasks Principal component analysis (PCA)


1.   Load or create a dataset with more than 2 dimensions.
2. Find the first 2 principal components with and without using sklearn.
3.  Practice using PCA to preserve a certain percentage of variance.
4. Train a classification or regression neural network by using:
  a.  The Original dataset
  b. Principal components
  
5. Repeat part 4 using Kernel PCA (linear, sigmoid, RBF).
6. Build a pipeline to tune the hyperparameters of Kernel PCA and also the neural network. Which hyperparameters can be tuned?

Sample codes for PCA:

https://github.com/ageron/handson-ml2/blob/master/08_dimensionality_reduction.ipynbLinks to an external site.

In [3]:
import sys
assert sys.version_info >= (3, 5)

# Scikit-Learn ≥0.20 is required
import sklearn
assert sklearn.__version__ >= "0.20"

# Common imports
import numpy as np
import os

# to make this notebook's output stable across runs
np.random.seed(42)

# To plot pretty figures
%matplotlib inline
import matplotlib as mpl
import matplotlib.pyplot as plt
mpl.rc('axes', labelsize=14)
mpl.rc('xtick', labelsize=12)
mpl.rc('ytick', labelsize=12)

# Where to save the figures
PROJECT_ROOT_DIR = "."
CHAPTER_ID = "dim_reduction"
IMAGES_PATH = os.path.join(PROJECT_ROOT_DIR, "images", CHAPTER_ID)
os.makedirs(IMAGES_PATH, exist_ok=True)

def save_fig(fig_id, tight_layout=True, fig_extension="png", resolution=300):
    path = os.path.join(IMAGES_PATH, fig_id + "." + fig_extension)
    print("Saving figure", fig_id)
    if tight_layout:
        plt.tight_layout()
    plt.savefig(path, format=fig_extension, dpi=resolution)

# Load or create a dataset with more than 2 dimensions: bulid a 4d datasets.

In [4]:
import numpy as np

# 构造一个简单的 4 维数据集
# 假设我们有 100 个样本，每个样本有 4 个特征（维度）
np.random.seed(42)
n_samples = 100

# 我们创建 4 个特征，前两个是比较重要维度（信号），后两个可以是一些相关性较低或有噪声
X1 = np.random.normal(loc=0.0, scale=1.0, size=n_samples)  # 维度 1
X2 = 2 * X1 + np.random.normal(0, 0.5, size=n_samples)  # 维度 2，与 X1 有线性关系
X3 = np.random.normal(0, 1, size=n_samples)          # 维度 3，比较随机
X4 = np.random.normal(5, 2, size=n_samples)     # 维度 4，另一个随机但平均值不同

# 把它们合并成一个 (100, 4) 的矩阵
X = np.vstack([X1, X2, X3, X4]).T  # 转置后，每一行是一个样本
print(X.shape)  # (100, 4)
# print(X)

(100, 4)


## 2 find the first two principal components without sklearn

In [5]:
# 手写 PCA
# 1. 标准化（零均值）
X_centered = X - np.mean(X, axis=0)

# 2. 计算协方差矩阵（特征之间的相关性）
cov_mat = np.cov(X_centered, rowvar=False)  # rowvar=False 表示每一列是一个变量（特征）

# 3. 求特征值和特征向量
eigenvalues, eigenvectors = np.linalg.eig(cov_mat)

# 特征值排序，从大到小
idx = np.argsort(eigenvalues)[::-1]
eigenvalues = eigenvalues[idx]
eigenvectors = eigenvectors[:, idx]

# 选前两个主成分（top-2 特征向量）
pc1 = eigenvectors[:, 0]
pc2 = eigenvectors[:, 1]
pc3 = eigenvectors [:,2]

print("eigenvalue:", eigenvalues[0], eigenvalues[1])
print("pc1:", pc1)
print("pc2:", pc2)
print("pc3", pc3)

# 投影：把原始数据映射到这两个主成分上
X_pca_manual = X_centered.dot(np.vstack([pc1, pc2]).T)
print(X_pca_manual.shape)  # 应该是 (100, 2)
print(X_pca_manual)

eigenvalue: 4.421142604131225 2.833970695693917
pc1: [ 0.39404577  0.80365295  0.11126027 -0.43184606]
pc2: [0.18656846 0.3784275  0.10231893 0.90084165]
pc3 [-0.05396152 -0.13437637  0.98845231 -0.04464508]
(100, 2)
[[ 1.46509767e+00 -1.36155514e+00]
 [ 3.84402496e-01 -1.26730488e+00]
 [ 9.17537174e-01  1.89802004e+00]
 [ 2.59974384e+00  2.38719673e+00]
 [-3.84732022e-01 -5.35424238e-01]
 [-2.27982043e-01 -1.34390058e-01]
 [ 3.15620092e+00  4.09602280e+00]
 [ 2.45808370e+00 -3.61587093e-01]
 [-9.67384708e-01  5.38833002e-01]
 [ 1.94315469e+00  4.22316645e-01]
 [-1.16302045e+00 -1.23941796e+00]
 [-1.48147714e+00  1.54605933e+00]
 [ 1.85618013e-01  1.71903928e+00]
 [-3.18557119e+00  8.80009338e-02]
 [-4.40814419e+00  5.50694811e-01]
 [-6.53935686e-01 -4.63300419e-01]
 [-2.43203409e+00  8.21107860e-02]
 [ 6.84867521e-01 -6.13276380e-01]
 [-1.40811322e+00 -2.11411464e-01]
 [-2.11892948e+00 -1.52160493e+00]
 [ 3.70881851e+00  1.83860966e+00]
 [-1.25513118e+00  3.90775361e-01]
 [ 1.76579925

## 2 find the first two principal components with sklearn

In [6]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# 标准化
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 用 PCA
pca = PCA(n_components=2)
X_pca_skl = pca.fit_transform(X_scaled)

print("sklearn variance ratio:", pca.explained_variance_ratio_)
print("sum of variance ration:", sum(pca.explained_variance_ratio_))


sklearn variance ratio: [0.52110374 0.2499482 ]
sum of variance ration: 0.7710519444226177


## 3- Practice using PCA to preserve a certain percentage of variance.


In [7]:
pca_var90 = PCA(n_components=0.90)  # 0.90 表示 90%
X_pca_90 = pca_var90.fit_transform(X_scaled)

print("保留 90% 方差时，选了多少主成分:", pca_var90.n_components_)
print("实际累计方差比例:", sum(pca_var90.explained_variance_ratio_))
print("降维后数据 shape:", X_pca_90.shape)
# print(X_pca_90)

保留 90% 方差时，选了多少主成分: 3
实际累计方差比例: 0.9913881443415489
降维后数据 shape: (100, 3)


# 4  Train a classification or regression neural network by using:
   a) The Original dataset



In [8]:
# 构造标签 y：如果 X1 + X2 > 某个值，就算为 1，否则为 0
y = (X[:, 0] + X[:, 1] > 0).astype(int)  # 100 个样本的标签

In [9]:
from tensorflow import keras
from tensorflow.keras import layers

# 构建简单的神经网络
model_orig = keras.Sequential([
    layers.Dense(16, activation='relu', input_shape=(4,)),
    layers.Dense(8, activation='relu'),
    layers.Dense(1, activation='sigmoid')  # 二分类
])

model_orig.compile(optimizer='adam',
          loss='binary_crossentropy',
          metrics=['accuracy'])

# 训练
model_orig.fit(X, y, epochs=50, batch_size=16, validation_split=0.2)


Epoch 1/50


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 55ms/step - accuracy: 0.4156 - loss: 1.4214 - val_accuracy: 0.5000 - val_loss: 1.1303
Epoch 2/50
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.4582 - loss: 1.3066 - val_accuracy: 0.5000 - val_loss: 1.0602
Epoch 3/50
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.4790 - loss: 1.1463 - val_accuracy: 0.5000 - val_loss: 0.9967
Epoch 4/50
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.4747 - loss: 1.0539 - val_accuracy: 0.5000 - val_loss: 0.9378
Epoch 5/50
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.4208 - loss: 1.0672 - val_accuracy: 0.5000 - val_loss: 0.8832
Epoch 6/50
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.4842 - loss: 0.9423 - val_accuracy: 0.5000 - val_loss: 0.8339
Epoch 7/50
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.4373 - loss: 0.8854 - val_accuracy: 0.5000 - val_loss: 0.7874
Epoch 8/50
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.4616 - loss: 0.8023 - val_accuracy: 0.5000 - val_loss: 0.7450
Epoch 9/50


 ## b Principal components

In [10]:
# 用我们之前得到的 X_pca_skl（2 维）作为输入
model_pca = keras.Sequential([
    layers.Dense(16, activation='relu', input_shape=(2,)),
    layers.Dense(8, activation='relu'),
    layers.Dense(1, activation='sigmoid')
])

model_pca.compile(optimizer='adam',
          loss='binary_crossentropy',
          metrics=['accuracy'])

model_pca.fit(X_pca_skl, y, epochs=50, batch_size=16, validation_split=0.2)


Epoch 1/50
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 80ms/step - accuracy: 0.8627 - loss: 0.5615 - val_accuracy: 0.8000 - val_loss: 0.5841
Epoch 2/50
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.8729 - loss: 0.5482 - val_accuracy: 0.8000 - val_loss: 0.5728
Epoch 3/50
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.8979 - loss: 0.5291 - val_accuracy: 0.8000 - val_loss: 0.5615
Epoch 4/50
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.8606 - loss: 0.5298 - val_accuracy: 0.8000 - val_loss: 0.5502
Epoch 5/50
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.9438 - loss: 0.4908 - val_accuracy: 0.8000 - val_loss: 0.5385
Epoch 6/50
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.9151 - loss: 0.5012 - val_accuracy: 0.8000 - val_loss: 0.5270
Epoch 7/50
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.9125 - loss: 0.4719 - val_accuracy: 0.8000 - val_loss: 0.5149
Epoch 8/50
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.9516 - loss: 0.4483 - val_accuracy: 0.9000 - val_loss: 0.5027


# 5 Repeat part 4 using Kernel PCA (linear, sigmoid, RBF).


In [11]:
from sklearn.decomposition import KernelPCA
import tensorflow as tf

# RBF Kernel PCA（最常用）
kpca_rbf = KernelPCA(n_components=2, kernel='rbf', gamma=0.1)
X_kpca_rbf = kpca_rbf.fit_transform(X_scaled)

# Sigmoid Kernel PCA
kpca_sigmoid = KernelPCA(n_components=2, kernel='sigmoid', gamma=0.01, coef0=1)
X_kpca_sigmoid = kpca_sigmoid.fit_transform(X_scaled)

# Linear Kernel PCA（其实和普通 PCA 类似）
kpca_linear = KernelPCA(n_components=2, kernel='linear')
X_kpca_linear = kpca_linear.fit_transform(X_scaled)


## 6. Build a pipeline to tune the hyperparameters of Kernel PCA and also the neural network. Which hyperparameters can be tuned?


In [12]:
import tensorflow as tf
from sklearn.model_selection import train_test_split

def train_model_with_2D_data(X_2D, y, title=""):
    # 1. 划分数据
    X_train, X_test, y_train, y_test = train_test_split(
        X_2D, y, test_size=0.2, random_state=42
    )

    # 2. 建模
    model = tf.keras.Sequential([
        tf.keras.layers.Dense(16, activation='relu', input_shape=(2,)),
        tf.keras.layers.Dense(8, activation='relu'),
        tf.keras.layers.Dense(1, activation='sigmoid')
    ])

    model.compile(
        optimizer='adam',
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    # 3. 训练
    history = model.fit(
        X_train, y_train,
        epochs=30,
        batch_size=16,
        validation_split=0.2,
        verbose=0  # 不显示太多细节
    )

    # 4. 测试集评估
    test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)

    # 5. 训练集最终 loss/acc
    final_train_loss = history.history["loss"][-1]
    final_train_acc = history.history["accuracy"][-1]

    # 6. 验证集最终 loss/acc
    final_val_loss = history.history["val_loss"][-1]
    final_val_acc = history.history["val_accuracy"][-1]

    # 7. 全部打印出来
    print("\n========================")
    print(f" {title} — Results")
    print("========================")
    print(f"Train set Loss:      {final_train_loss:.4f}")
    print(f"Train set Accuracy:  {final_train_acc:.4f}")
    print(f"Val set Loss:      {final_val_loss:.4f}")
    print(f"Val set Accuracy:  {final_val_acc:.4f}")
    print(f"Test set Loss:      {test_loss:.4f}")
    print(f"Test set Accuracy:  {test_acc:.4f}")
    print("========================\n")

    return model, history


# 使用上面的通用函数 训练三个 Kernel PCA model

In [13]:
from sklearn.decomposition import KernelPCA

# 你的原始数据：X_scaled (n, 4)

# 1️⃣ RBF Kernel PCA
kpca_rbf = KernelPCA(n_components=2, kernel='rbf', gamma=0.1)
X_kpca_rbf = kpca_rbf.fit_transform(X_scaled)

# 2️⃣ Sigmoid Kernel PCA
kpca_sigmoid = KernelPCA(n_components=2, kernel='sigmoid', gamma=0.01, coef0=1)
X_kpca_sigmoid = kpca_sigmoid.fit_transform(X_scaled)

# 3️⃣ Linear Kernel PCA
kpca_linear = KernelPCA(n_components=2, kernel='linear')
X_kpca_linear = kpca_linear.fit_transform(X_scaled)


In [14]:
train_model_with_2D_data(X_kpca_rbf, y, "RBF Kernel PCA")
train_model_with_2D_data(X_kpca_sigmoid, y, "Sigmoid Kernel PCA")
train_model_with_2D_data(X_kpca_linear, y, "Linear Kernel PCA")

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)



 RBF Kernel PCA — Results
Train set Loss:      0.4903
Train set Accuracy:  0.9219
Val set Loss:      0.4504
Val set Accuracy:  1.0000
Test set Loss:      0.4935
Test set Accuracy:  0.8500


 Sigmoid Kernel PCA — Results
Train set Loss:      0.5814
Train set Accuracy:  0.9688
Val set Loss:      0.5483
Val set Accuracy:  0.8750
Test set Loss:      0.6012
Test set Accuracy:  0.8500


 Linear Kernel PCA — Results
Train set Loss:      0.3308
Train set Accuracy:  0.9688
Val set Loss:      0.2845
Val set Accuracy:  0.8750
Test set Loss:      0.3899
Test set Accuracy:  0.8500



(<Sequential name=sequential_4, built=True>,
 <keras.src.callbacks.history.History at 0x7ec3b699a930>)

In [1]:

%pip install scikeras
import numpy as np
import os
from sklearn.decomposition import KernelPCA
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from scikeras.wrappers import KerasClassifier
import tensorflow as tf

# 1. 构建神经网络模型函数（KerasClassifier 需要）
# The create_model function should accept 'meta' dictionary from Scikeras
def create_model(meta, hidden_units=[16,8], activation='relu', optimizer='adam'):
    # Extract input_dim from meta, which is provided by Scikeras in a Pipeline context
    input_dim = meta["n_features_in_"]

    model = tf.keras.Sequential()
    model.add(tf.keras.layers.Input(shape=(input_dim,))) # Use the dynamic input_dim
    for units in hidden_units:
        model.add(tf.keras.layers.Dense(units, activation=activation))
    model.add(tf.keras.layers.Dense(1, activation='sigmoid'))
    model.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])
    return model

# 2. 包装成 KerasClassifier
kera_clf = KerasClassifier(model=create_model, verbose=0, random_state=42)

# 3. 建立 Pipeline
pipe = Pipeline([
    ('kpca', KernelPCA()),
    ('nn', kera_clf)
])

# 4. 定义超参数搜索空间
param_grid = {
    'kpca__n_components': [2, 3],
    'kpca__kernel': ['linear', 'rbf', 'sigmoid'],
    'kpca__gamma': [0.01, 0.1, 0.5], # Gamma is only used for RBF kernel
    'kpca__coef0': [0, 1],           # Coef0 is only used for Sigmoid kernel
    'nn__model__hidden_units': [[16,8], [32,16,8]],
    'nn__model__activation': ['relu', 'tanh'],
    'nn__optimizer': ['adam', 'sgd'],
    'nn__batch_size': [16, 32],
    'nn__epochs': [30, 50]
}

# 5. Grid Search
grid = GridSearchCV(pipe, param_grid, cv=3, scoring='accuracy', error_score='raise')
grid.fit(X_scaled, y)

print("最佳超参数组合：", grid.best_params_)
print("最佳准确率：", grid.best_score_)


  Using cached scikit_learn-1.3.2-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (11 kB)
Using cached scikit_learn-1.3.2-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (10.8 MB)
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.8.0
    Uninstalling scikit-learn-1.8.0:
      Successfully uninstalled scikit-learn-1.8.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
scikeras 0.13.0 requires scikit-learn>=1.4.2, but you have scikit-learn 1.3.2 which is incompatible.
umap-learn 0.5.9.post2 requires scikit-learn>=1.6, but you have scikit-learn 1.3.2 which is incompatible.
spopt 0.7.0 requires scikit-learn>=1.4.0, but you have scikit-learn 1.3.2 which is incompatible.
imbalanced-learn 0.14.0 requires scikit-learn<2,>=1.4.2, but you have scikit-learn 1.3.2 which is incompatible.
shap 0.50.0 requires n

NameError: name 'X_scaled' is not defined

# 2 Task Classification problem
1. Load IRIS and MNIST fashion datasets from Keras.
2. Build and train a neural network for classifying the loaded labeled datasets.
3. Tune the hyperparameters (including hidden layer size and activation functions). What else can be tuned?
4. Plot the loss and accuracy for training and testing datasets.
5. Save the weights of the layers and use callbacks during the training process.
6. Practice saving and loading the trained model.

## 1. Load IRIS and MNIST fashion datasets from Keras.

In [ ]:
# Python ≥3.5 is required
import sys
assert sys.version_info >= (3, 5)

# Scikit-Learn ≥0.20 is required
import sklearn
assert sklearn.__version__ >= "0.20"

try:
    # %tensorflow_version only exists in Colab.
    %tensorflow_version 2.x
except Exception:
    pass

# TensorFlow ≥2.0 is required
import tensorflow as tf
assert tf.__version__ >= "2.0"

# Keras
from tensorflow.keras.datasets import fashion_mnist
from sklearn.datasets import load_iris

# Common imports
import numpy as np
import os

# to make this notebook's output stable across runs
np.random.seed(42)

# To plot pretty figures
%matplotlib inline
import matplotlib as mpl
import matplotlib.pyplot as plt
mpl.rc('axes', labelsize=14)
mpl.rc('xtick', labelsize=12)
mpl.rc('ytick', labelsize=12)

# Where to save the figures
PROJECT_ROOT_DIR = "."
CHAPTER_ID = "ann"
IMAGES_PATH = os.path.join(PROJECT_ROOT_DIR, "images", CHAPTER_ID)
os.makedirs(IMAGES_PATH, exist_ok=True)

def save_fig(fig_id, tight_layout=True, fig_extension="png", resolution=300):
    path = os.path.join(IMAGES_PATH, fig_id + "." + fig_extension)
    print("Saving figure", fig_id)
    if tight_layout:
        plt.tight_layout()
    plt.savefig(path, format=fig_extension, dpi=resolution)

In [ ]:
# Load MNIST Fashion dataset
(x_train_fashion, y_train_fashion), (x_test_fashion, y_test_fashion) = fashion_mnist.load_data()

# Load IRIS dataset
iris_data = load_iris()
X_iris = iris_data.data
y_iris = iris_data.target

# Displaying shapes of the datasets
print(f"Fashion MNIST Training Data Shape: {x_train_fashion.shape}")
print(f"Fashion MNIST Training Labels Shape: {y_train_fashion.shape}")
print(f"Fashion MNIST Test Data Shape: {x_test_fashion.shape}")
print(f"Fashion MNIST Test Labels Shape: {y_test_fashion.shape}")

print(f"IRIS Data Shape: {X_iris.shape}")
print(f"IRIS Labels Shape: {y_iris.shape}")

# 2- Build and train a neural network for classifying the loaded labeled datasets.

In [ ]:
# 导入必要的库
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split

# 1. Fashion MNIST 数据集

# 定义输入形状
input_shape_fashion = (28, 28, 1)  # Fashion MNIST 是28x28的灰度图像

# 归一化（将数据缩放到0-1范围内）
x_train_fashion = x_train_fashion / 255.0
x_test_fashion = x_test_fashion / 255.0

# 调整数据形状以适应卷积神经网络（CNN）
x_train_fashion = x_train_fashion.reshape(-1, 28, 28, 1)
x_test_fashion = x_test_fashion.reshape(-1, 28, 28, 1)

# 创建 Fashion MNIST 神经网络模型
model_fashion = models.Sequential([
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=input_shape_fashion),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dense(10, activation='softmax')  # 输出10个类别（0到9）
])

# 编译模型
model_fashion.compile(optimizer='adam',
                      loss='sparse_categorical_crossentropy',
                      metrics=['accuracy'])

# 训练模型
model_fashion.fit(x_train_fashion, y_train_fashion, epochs=5, batch_size=64, validation_split=0.2)

# 评估模型
test_loss, test_acc = model_fashion.evaluate(x_test_fashion, y_test_fashion)
print(f"Fashion MNIST 模型在测试集上的准确率: {test_acc}")

# 2. IRIS 数据集

# 数据预处理：IRIS 数据集是一个4维特征的分类问题，目标是预测三种花的类别（0, 1, 2）
X_train_iris, X_test_iris, y_train_iris, y_test_iris = train_test_split(X_iris, y_iris, test_size=0.2, random_state=42)

# 创建 IRIS 数据集神经网络模型
model_iris = models.Sequential([
    layers.Dense(64, activation='relu', input_dim=X_train_iris.shape[1]),  # 输入层，64个神经元
    layers.Dense(32, activation='relu'),  # 隐藏层，32个神经元
    layers.Dense(3, activation='softmax')  # 输出层，3个神经元，对应三种类别
])

# 编译模型
model_iris.compile(optimizer='adam',
                   loss='sparse_categorical_crossentropy',
                   metrics=['accuracy'])

# 训练模型
model_iris.fit(X_train_iris, y_train_iris, epochs=50, batch_size=16, validation_split=0.2)

# 评估模型
test_loss_iris, test_acc_iris = model_iris.evaluate(X_test_iris, y_test_iris)
print(f"IRIS model testset accuracy: {test_acc_iris}")


## 3. Tune the hyperparameters (including hidden layer size and activation functions). What else can be tuned?


1. layers quanitiy and size
2. Activation function  : relu, Sigmod, Tanh
3. Optimizer: LR, Adam,SGD,RMSprop
4. Batch size
5. Epochs
6. Regulazation : dropout, L2 regulazation.
7. Batch Normalization

## 1. Fashion MNIST hyperparameter finetune

In [ ]:
# 创建调整后的Fashion MNIST神经网络模型
model_fashion_tuned = models.Sequential([
    layers.Conv2D(64, (3, 3), activation='relu', input_shape=input_shape_fashion),  # 增加卷积层的滤波器数量
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(128, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(128, (3, 3), activation='relu'),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),  # 增加隐藏层的大小
    layers.Dropout(0.5),  # 添加Dropout层以防止过拟合
    layers.Dense(10, activation='softmax')  # 输出层
])

# 编译模型（调整优化器学习率）
model_fashion_tuned.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),  # 调整学习率
                loss='sparse_categorical_crossentropy',
                metrics=['accuracy'])

# 训练模型（调整批量大小和轮数）
model_fashion_tuned.fit(x_train_fashion, y_train_fashion, epochs=4, batch_size=32, validation_split=0.2)

# 评估模型
test_loss, test_acc = model_fashion_tuned.evaluate(x_test_fashion, y_test_fashion)
print(f"调优后的Fashion MNIST模型在测试集上的准确率: {test_acc}")


## 2. IRIS hypermeer finetuning

In [ ]:
# 创建调整后的IRIS神经网络模型
model_iris_tuned = models.Sequential([
    layers.Dense(128, activation='relu', input_dim=X_train_iris.shape[1]),  # 增加神经元数量
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.4),  # 添加Dropout层防止过拟合
    layers.Dense(3, activation='softmax')  # 输出层
])

# 编译模型（调整优化器学习率）
model_iris_tuned.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.005),  # 调整学习率
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

# 训练模型（调整批量大小和轮数）
model_iris_tuned.fit(X_train_iris, y_train_iris, epochs=100, batch_size=32, validation_split=0.2)

# 评估模型
test_loss_iris, test_acc_iris = model_iris_tuned.evaluate(X_test_iris, y_test_iris)
print(f"调优后的IRIS模型在测试集上的准确率: {test_acc_iris}")
